# Thêm Thư Viện

In [1]:
import pyodbc
import pandas as pd
import numpy as np

# Tạo kết nối

In [2]:
conn_dwh_library = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;' # Địa chỉ IP của SQL Server
    'DATABASE=dwh_library;' # Tên cơ sở dữ liệu
    'UID=itc;'              # Tên đăng nhập
    'PWD=spkt@2024;'
)

## Đọc data từ SQL Server

In [6]:
query_phieumuon = """SELECT PMS.ID_phieu_muon, PMS.ID_ban_doc, ID_thu_vien, ID_nhom_ban_doc, Ngay_muon 
                        FROM DIM_Phieu_muon_sach PMS
                                JOIN DIM_Ban_doc BD ON PMS.ID_ban_doc = BD.ID_ban_doc
                                JOIN DIM_Xep_gia XG ON PMS.ID_xep_gia =  XG.ID_xep_gia
                        WHERE Ngay_muon >= 20040101 AND Ngay_muon < 20050101"""
df_phieumuon = pd.read_sql(query_phieumuon, conn_dwh_library)
print(df_phieumuon)

C:\Users\phung\AppData\Local\Temp\ipykernel_21020\4080579443.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_phieumuon = pd.read_sql(query_phieumuon, conn_dwh_library)


       ID_phieu_muon ID_ban_doc ID_thu_vien  ID_nhom_ban_doc  Ngay_muon
0             156308          0      DHSPKT                0   20040515
1             195732          0      DHSPKT                0   20041102
2             163609          0      DHSPKT                0   20040610
3             169000          0      DHSPKT                0   20040719
4             170271          0      DHSPKT                0   20040727
...              ...        ...         ...              ...        ...
98152         179940          0      DHSPKT                0   20040921
98153         177606          0      DHSPKT                0   20040920
98154         178211   04102023      DHSPKT               15   20040921
98155         180373          0      DHSPKT                0   20040921
98156         179062          0      DHSPKT                0   20040921

[98157 rows x 5 columns]


# Xử lý code

## Lượng người dùng theo thư viện, nhóm bạn, ngày mượn

In [7]:
so_luong_muon_sach = df_phieumuon.groupby(['ID_thu_vien', 'ID_nhom_ban_doc', 'Ngay_muon'])['ID_ban_doc'].count().reset_index()
so_luong_muon_sach = so_luong_muon_sach.rename(columns={'ID_ban_doc': 'So_nguoi_dung'})
print(so_luong_muon_sach)

    ID_thu_vien  ID_nhom_ban_doc  Ngay_muon  So_nguoi_dung
0             0                0   20040101              7
1             0                0   20040102             23
2             0                0   20040103             10
3             0                0   20040105             14
4             0                0   20040106             10
..          ...              ...        ...            ...
952      DHSPKT               24   20041207              1
953      DHSPKT               24   20041214              3
954      DHSPKT               24   20041222              3
955      DHSPKT               24   20041223              1
956      DHSPKT               24   20041231              1

[957 rows x 4 columns]


## Load data

### [Nếu cần] Clear bảng

In [10]:
cursor = conn_dwh_library.cursor()
truncate_query = "DELETE FROM FACT_Thu_vien"
cursor.execute(truncate_query)
conn_dwh_library.commit()
cursor.close()

### Load data vào bảng Dim

In [11]:
cursor_dwh = conn_dwh_library.cursor()
insert_query = """
                INSERT INTO FACT_Thu_vien (ID_thu_vien, 
                                            ID_nhom_ban_doc,  
                                            ID_date, 
                                            So_nguoi_dung)
                VALUES (?, ?, ?, ?)
                """
for index, row in so_luong_muon_sach.iterrows():
   # Trích xuất giá trị từ các cột
    values = (row['ID_thu_vien'],
              row['ID_nhom_ban_doc'],
              row['Ngay_muon'],
              row['So_nguoi_dung'])  # Nếu cột này có tên đúng
    cursor_dwh.execute(insert_query, values)
conn_dwh_library.commit()